# Generating new data 

This notebook generates two synthetic enrichment tables:

1. **`factMarketSignals.csv`** at `year_month × product_group × region_id` grain.
2. **`factMarketActivities.csv`** at `year_month × product_id × region_id` grain.

The tables simulate external market conditions and product-level marketing engagement while remaining aligned with the existing sales, product, customer, inventory, CRM pipeline and regional data.

## Business purpose and modeling boundary

The project does not contain real external market-research or digital-marketing feeds. These two tables therefore provide a transparent **synthetic enrichment layer** for portfolio demonstration.

| Output | Grain | Purpose |
|---|---|---|
| `market_signals` | Month × product group × region | Demand, competition, macro conditions, supply pressure, and market opportunity |
| `market_activity` | Month × product × region | Campaign spend, channel, impressions, clicks, website engagement, demos, and leads |

The generated values are not presented as real observations. They are calibrated with existing business structure and operational drivers so that relationships are plausible and reproducible.

**Prediction timing assumption:** the Early-Warning model runs after month-end. Same-month inventory, CRM, pipeline information is therefore available. Internal sales are used only as a **lagged calibration driver**, not as a future input.


## Notebook roadmap

1. Configure reproducible paths and random seeds
2. Load existing project tables
3. Inspect schemas and validate required fields
4. Standardize dates, numeric fields, IDs, product-group labels
5. Restrict generation to the observed sales horizon
6. Build complete CRM, pipeline, inventory, lagged-sales calibration panels
7. Generate group-level market signals
8. Build an active product–region–month panel that excludes pre-launch months
9. Generate product-level marketing activity
10. Validate keys, ranges, business rules and join coverage
11. Export full datasets, dictionaries, metadata and GitHub samples.

## 1. Setup 

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

RANDOM_SEED = 42
MARKET_SIGNAL_SEED = RANDOM_SEED + 20000
MARKET_ACTIVITY_SEED = RANDOM_SEED + 20001

INPUT_DIR = Path("../data/full_data/sql_input")
OUTPUT_DIR = Path("../data/full_data/new_generated_data")
SAMPLE_OUTPUT_DIR = Path("../data/samples/02_new_generated_data")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input sql data directory: {INPUT_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Output directory sample: {SAMPLE_OUTPUT_DIR.resolve()}")

Input sql data directory: C:\Benutzer\Anastasia\Lokale Daten\AS Portfolio\early-warning\data\full_data\sql_input
Output directory: C:\Benutzer\Anastasia\Lokale Daten\AS Portfolio\early-warning\data\full_data\new_generated_data
Output directory sample: C:\Benutzer\Anastasia\Lokale Daten\AS Portfolio\early-warning\data\samples\02_new_generated_data


## 2. Load existing data

In [2]:
REQUIRED_FILES = {
    "dates": "dates.csv",
    "regions": "regions.csv",
    "products": "products.csv",
    "customers": "customers.csv",
    "sales": "sales.csv",
    "inventory": "inventory.csv",
    "crm_activities": "crm_activities.csv",
    "pipeline": "pipeline.csv"}

OPTIONAL_FILES = {
    "costs": "costs.csv",
    "returns": "returns.csv",
    "sales_reps": "sales_reps.csv"}

# Load required project CSVs and any optional context tables that exist
def load_project_tables(
    data_dir: Path, 
    required_files: dict[str, str],
    optional_files: dict[str, str]) -> dict[str, pd.DataFrame]:
    missing = [
        filename
        for filename in required_files.values()
        if not (data_dir / filename).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing required files in {data_dir.resolve()}: {missing}")

    tables = {
        name: pd.read_csv(data_dir / filename, low_memory=False)
        for name, filename in required_files.items()}
    for name, filename in optional_files.items():
        path = data_dir / filename
        if path.exists():
            tables[name] = pd.read_csv(path, low_memory=False)

    return tables


raw = load_project_tables(INPUT_DIR, REQUIRED_FILES, OPTIONAL_FILES)
print(f"Loaded {len(raw)} project tables.")

Loaded 11 project tables.


In [3]:
# Check table inentory 
table_overview = pd.DataFrame([
    {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells_pct": round(df.isna().mean().mean() * 100, 2),
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1_048_576, 2)}
    for name, df in raw.items()
]).sort_values("table").reset_index(drop=True)

display(table_overview)

,table,rows,columns,duplicate_rows,missing_cells_pct,memory_mb
0,costs,12000,7,0,0.000,1.220
1,crm_activities,35000,10,0,0.000,7.520
2,customers,1200,9,0,0.000,0.380
3,dates,4018,18,0,0.000,1.990
4,inventory,69852,10,0,0.000,8.260
5,pipeline,15000,14,0,0.000,4.580
6,products,200,14,0,0.000,0.080
7,regions,10,7,0,0.000,0.000
8,returns,4200,10,0,0.000,0.740
9,sales,120000,26,0,0.100,57.750


In [4]:
source_columns = pd.DataFrame([
    {"table": name, "columns": ", ".join(df.columns)}
    for name, df in raw.items()])
display(source_columns)

for table_name in ["products", "sales", "inventory", "crm_activities", "pipeline"]:
    print(f"\n{table_name.upper()} — first 3 rows")
    display(raw[table_name].head(3))

,table,columns
0,dates,"DateKey, Date, Year, Quarter, QuarterName, Mon..."
1,regions,"region_id, country, region, currency, fx_to_eu..."
2,products,"product_id, sku, product_family, product_group..."
3,customers,"customer_id, customer_code, customer_segment, ..."
4,sales,"sales_id, order_id, date_id, order_date, year_..."
5,inventory,"inventory_id, year_month, product_id, region_i..."
6,crm_activities,"activity_id, date_id, activity_date, customer_..."
7,pipeline,"opportunity_id, created_date, customer_id, pro..."
8,costs,"cost_id, year_month, product_id, standard_unit..."
9,returns,"return_id, sales_id, date_id, return_date, cus..."



PRODUCTS — first 3 rows


,product_id,sku,product_family,product_group,product_name,launch_year,lifecycle_stage,base_list_price_eur,base_unit_cost_eur,target_margin_pct,product_growth_factor,is_declining_product,is_new_product,launch_date
0,1000,SKU-1000,IT Devices,Laptop Pro,Laptop Pro Model 01,2022,Decline,804.370,540.920,0.290,1.080,1,0,2022-01-01
1,1001,SKU-1001,IT Devices,Laptop Pro,Laptop Pro Model 02,2023,Growth,771.330,542.710,0.290,1.080,0,0,2023-01-01
2,1002,SKU-1002,IT Devices,Laptop Pro,Laptop Pro Model 03,2024,Growth,828.800,552.000,0.290,1.080,0,0,2024-01-01



SALES — first 3 rows


,sales_id,order_id,date_id,order_date,year_month,customer_id,product_id,region_id,sales_rep_id,units,asp_eur,discount_pct,revenue_eur,revenue_local_currency,currency,unit_cost_eur,gross_profit_eur,gross_margin_pct,is_outlier_order,cogs_eur,discount_value_eur,margin_category,order_size_category,missing_sales_rep_flag,missing_discount_flag,missing_margin_flag
0,1,ORD-00000001,20220304,2022-03-04,2022-03-01,282,1133,2,39.000,26,104.000,0.092,"2,704.130","2,704.130",EUR,78.470,663.790,0.245,False,"2,040.340",248.780,Low Margin,Medium Order,0,0,0
1,2,ORD-00000002,20240503,2024-05-03,2024-05-01,562,1105,2,73.000,21,"1,698.070",0.089,"35,659.450","35,659.450",EUR,944.570,"15,823.420",0.444,False,"19,836.030","3,173.691",High Margin,Small Order,0,0,0
2,3,ORD-00000003,20241205,2024-12-05,2024-12-01,146,1007,7,41.000,117,983.150,0.123,"115,029.000","169,160.290",CAD,637.920,"40,392.640",0.351,False,"74,636.360","14,148.567",Medium Margin,Large Order,0,0,0


...

,inventory_id,year_month,product_id,region_id,opening_stock_units,production_units,ending_stock_units,stockout_flag,inventory_value_eur,zero_stock_flag
0,1,2021-01-01,1000,1,53,79,51,False,"30,108.390",0
1,2,2021-01-01,1000,3,31,22,34,False,"11,838.230",0
2,3,2021-01-01,1000,7,15,13,18,False,"7,186.070",0



CRM_ACTIVITIES — first 3 rows


,activity_id,date_id,activity_date,customer_id,sales_rep_id,activity_type,activity_minutes,sentiment_score,customer_health_score,customer_health_band
0,1,20241105,2024-11-05,1128,57,Call,22,1.386,90.300,Healthy
1,2,20221123,2022-11-23,1049,60,Email,40,-0.289,50.900,Neutral
2,3,20240619,2024-06-19,1006,48,Email,18,0.337,73.700,Neutral



PIPELINE — first 3 rows


,opportunity_id,created_date,customer_id,product_group,sales_rep_id,stage,expected_value_eur,win_probability,expected_close_date,created_date_id,weighted_pipeline_eur,days_to_close,is_closed_won,is_closed_lost
0,1,2021-08-12,56,Connectivity Module,18,Lead,"137,697.630",0.413,2022-03-15,20210812,"56,869.120",215,0,0
1,2,2021-02-14,1038,Accessory Kit,41,Qualified,"109,985.570",0.452,2021-07-09,20210214,"49,713.480",145,0,0
2,3,2022-07-09,546,Service Contract,59,Qualified,"31,257.790",0.153,2022-12-06,20220709,"4,782.440",150,0,0


## 3. Validate the actual source schema

In [5]:
REQUIRED_COLUMNS = {
    "dates": {
        "Date", "YearMonth"},
    "regions": {
        "region_id", "region", "market_growth_factor"},
    "products": {
        "product_id", "product_family", "product_group",
        "lifecycle_stage", "base_list_price_eur",
        "target_margin_pct", "product_growth_factor", "launch_date"},
    "customers": {
        "customer_id", "region_id"},
    "sales": {
        "sales_id", "year_month", "customer_id", "product_id",
        "region_id", "units", "revenue_eur"},
    "inventory": {
        "year_month", "product_id", "region_id",
        "opening_stock_units", "production_units",
        "ending_stock_units", "stockout_flag"},
    "crm_activities": {
        "activity_id", "activity_date", "customer_id",
        "activity_minutes", "customer_health_score"},
    "pipeline": {
        "opportunity_id", "created_date", "customer_id",
        "product_group", "weighted_pipeline_eur", "win_probability"}}

# Return one row per missing table or field
def find_schema_issues(
    datasets: dict[str, pd.DataFrame],
    required_columns: dict[str, set[str]],
) -> pd.DataFrame:
    issues = []
    for table_name, expected in required_columns.items():
        if table_name not in datasets:
            issues.append({
                "table": table_name,
                "missing_column": "<table missing>"})
            continue
        for column in sorted(expected - set(datasets[table_name].columns)):
            issues.append({
                "table": table_name,
                "missing_column": column})
    return pd.DataFrame(issues, columns=["table", "missing_column"])


schema_issues = find_schema_issues(raw, REQUIRED_COLUMNS)
if not schema_issues.empty:
    display(schema_issues)
    raise ValueError("Source schema validation failed.")

print("Schema validation passed.")


Schema validation passed.


## 4. Standardization helpers

In [6]:
# Convert dates, YYYYMM keys, or month strings to canonical YYYY-MM text
def normalize_year_month(
    series: pd.Series,
    column_name: str,
) -> pd.Series:
    raw_values = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True))
    parsed = pd.Series(
        pd.NaT,
        index=series.index,
        dtype="datetime64[ns]")

    yyyymm_mask = raw_values.str.fullmatch(r"\d{6}", na=False)
    parsed.loc[yyyymm_mask] = pd.to_datetime(
        raw_values.loc[yyyymm_mask],
        format="%Y%m",
        errors="coerce")

    year_month_mask = raw_values.str.fullmatch(
        r"\d{4}-\d{2}", na=False)
    parsed.loc[year_month_mask] = pd.to_datetime(
        raw_values.loc[year_month_mask],
        format="%Y-%m",
        errors="coerce")

    remaining_mask = ~(yyyymm_mask | year_month_mask)
    parsed.loc[remaining_mask] = pd.to_datetime(
        raw_values.loc[remaining_mask],
        errors="coerce")

    if parsed.isna().any():
        invalid_examples = (
            raw_values.loc[parsed.isna()]
            .drop_duplicates()
            .head(10)
            .tolist())
        raise ValueError(
            f"{column_name} contains invalid date values: {invalid_examples}")

    return parsed.dt.to_period("M").astype(str)

# Convert regular numbers, decimal-comma values, currencies, and percentages
def convert_to_numeric(
    series: pd.Series,
    percent: bool = False,
) -> pd.Series:
    values = series.astype("string").str.strip()
    percent_mask = values.str.contains("%", na=False)
    values = values.str.replace(r"[€$£\s]", "", regex=True)

    comma_position = values.str.rfind(",")
    dot_position = values.str.rfind(".")
    has_comma = values.str.contains(",", regex=False, na=False)
    has_dot = values.str.contains(".", regex=False, na=False)
    has_both = has_comma & has_dot

    european = has_both & (comma_position > dot_position)
    values.loc[european] = (
        values.loc[european]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False))

    international = has_both & ~european
    values.loc[international] = (
        values.loc[international]
        .str.replace(",", "", regex=False))

    comma_only = has_comma & ~has_dot
    values.loc[comma_only] = values.loc[comma_only].str.replace(
        ",", ".", regex=False)

    numeric_values = pd.to_numeric(values, errors="coerce")
    if percent:
        numeric_values.loc[percent_mask] = (
            numeric_values.loc[percent_mask] / 100)
    return numeric_values


# Convert integer-like IDs to a nullable integer representation
def normalize_integer_key(
    series: pd.Series,
    column_name: str,
) -> pd.Series:
    numeric_values = pd.to_numeric(series, errors="coerce")
    invalid = numeric_values.isna() & series.notna()
    if invalid.any():
        examples = series.loc[invalid].drop_duplicates().head(10).tolist()
        raise ValueError(f"{column_name} contains invalid IDs: {examples}")
    return numeric_values.astype("Int64")

# Trim labels and collapse repeated spaces while preserving business casing
def normalize_label(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True))

## 5. Standardize source tables

In [7]:
dates = raw["dates"].copy()
regions = raw["regions"].copy()
products = raw["products"].copy()
customers = raw["customers"].copy()
sales = raw["sales"].copy()
inventory = raw["inventory"].copy()
crm = raw["crm_activities"].copy()
pipeline = raw["pipeline"].copy()

# Canonical IDs
for table_name, frame, columns in [
    ("regions", regions, ["region_id"]),
    ("products", products, ["product_id"]),
    ("customers", customers, ["customer_id", "region_id"]),
    (
        "sales",
        sales,
        ["sales_id", "customer_id", "product_id", "region_id"]),
    ("inventory", inventory, ["product_id", "region_id"]),
    ("crm", crm, ["activity_id", "customer_id"]),
    ("pipeline", pipeline, ["opportunity_id", "customer_id"]),
]:
    for column in columns:
        frame[column] = normalize_integer_key(
            frame[column],
            f"{table_name}.{column}")

# Canonical months
dates["year_month"] = normalize_year_month(
    dates["YearMonth"], "dates.YearMonth")
sales["year_month"] = normalize_year_month(
    sales["year_month"], "sales.year_month")
inventory["year_month"] = normalize_year_month(
    inventory["year_month"], "inventory.year_month")
crm["year_month"] = normalize_year_month(
    crm["activity_date"], "crm_activities.activity_date")
pipeline["year_month"] = normalize_year_month(
    pipeline["created_date"], "pipeline.created_date")

# Canonical labels and dates
products["product_family"] = normalize_label(products["product_family"])
products["product_group"] = normalize_label(products["product_group"])
products["lifecycle_stage"] = normalize_label(products["lifecycle_stage"])
products["launch_date"] = pd.to_datetime(
    products["launch_date"], errors="coerce")
products["launch_month"] = products["launch_date"].dt.to_period("M").astype(str)
pipeline["product_group_raw"] = normalize_label(pipeline["product_group"])

# Numeric driver columns
products["base_list_price_eur"] = convert_to_numeric(
    products["base_list_price_eur"])
products["target_margin_pct"] = convert_to_numeric(
    products["target_margin_pct"], percent=True)
products["product_growth_factor"] = convert_to_numeric(
    products["product_growth_factor"])
regions["market_growth_factor"] = convert_to_numeric(
    regions["market_growth_factor"])
sales["units"] = convert_to_numeric(sales["units"])
sales["revenue_eur"] = convert_to_numeric(sales["revenue_eur"])
crm["activity_minutes"] = convert_to_numeric(crm["activity_minutes"])
crm["customer_health_score"] = convert_to_numeric(
    crm["customer_health_score"])
pipeline["weighted_pipeline_eur"] = convert_to_numeric(
    pipeline["weighted_pipeline_eur"])
pipeline["win_probability"] = convert_to_numeric(
    pipeline["win_probability"], percent=True)
for column in ["opening_stock_units", "production_units", "ending_stock_units"]:
    inventory[column] = convert_to_numeric(inventory[column])


### Preserve canonical product-group labels

Joins should use the same capitalization as the product dimension. Instead of converting the final output to lowercase, we create a case-insensitive lookup key internally and map pipeline labels back to the canonical product-group name.

In [8]:
product_group_lookup = (
    products[["product_group"]]
    .drop_duplicates()
    .assign(
        product_group_key=lambda frame: (
            frame["product_group"].str.casefold())))

duplicate_group_keys = product_group_lookup.duplicated(
    "product_group_key", keep=False)
if duplicate_group_keys.any():
    display(product_group_lookup.loc[duplicate_group_keys])
    raise ValueError(
        "Several canonical product groups collapse to the same "
        "case-insensitive key.")

pipeline["product_group_key"] = pipeline[
    "product_group_raw"
].str.casefold()
pipeline = pipeline.drop(columns=["product_group"], errors="ignore").merge(
    product_group_lookup,
    on="product_group_key",
    how="left",
    validate="m:1")

unknown_pipeline_groups = (
    pipeline.loc[pipeline["product_group"].isna(), "product_group_raw"]
    .drop_duplicates()
    .tolist())
if unknown_pipeline_groups:
    raise ValueError(
        "Pipeline contains product groups that are not present in products: "
        f"{unknown_pipeline_groups[:10]}")

print("Pipeline product groups map to canonical product labels.")

Pipeline product groups map to canonical product labels.


In [9]:
# Check standardized values and integrity

key_checks = pd.Series({
    "unique_region_id": regions["region_id"].is_unique,
    "unique_product_id": products["product_id"].is_unique,
    "unique_customer_id": customers["customer_id"].is_unique,
    "unique_sales_id": sales["sales_id"].is_unique,
    "sales_products_known": sales["product_id"].isin(products["product_id"]).all(),
    "sales_customers_known": sales["customer_id"].isin(
        customers["customer_id"]
    ).all(),
    "sales_regions_known": sales["region_id"].isin(
        regions["region_id"]
    ).all(),
    "inventory_products_known": inventory["product_id"].isin(
        products["product_id"]
    ).all(),
    "inventory_regions_known": inventory["region_id"].isin(
        regions["region_id"]
    ).all(),
    "crm_customers_known": crm["customer_id"].isin(
        customers["customer_id"]
    ).all(),
    "pipeline_customers_known": pipeline["customer_id"].isin(
        customers["customer_id"]
    ).all(),
}, name="passed")
display(key_checks.rename_axis("check").reset_index())

if not key_checks.all():
    raise AssertionError("At least one key-integrity check failed.")

numeric_quality = pd.DataFrame([
    {
        "column": label,
        "missing_after_conversion": int(series.isna().sum()),
        "minimum": series.min(),
        "maximum": series.max(),
    }
    for label, series in {
        "products.base_list_price_eur": products["base_list_price_eur"],
        "products.target_margin_pct": products["target_margin_pct"],
        "products.product_growth_factor": products["product_growth_factor"],
        "regions.market_growth_factor": regions["market_growth_factor"],
        "sales.units": sales["units"],
        "crm.activity_minutes": crm["activity_minutes"],
        "crm.customer_health_score": crm["customer_health_score"],
        "pipeline.weighted_pipeline_eur": pipeline["weighted_pipeline_eur"],
        "pipeline.win_probability": pipeline["win_probability"]}.items()])
display(numeric_quality)


,check,passed
0,unique_region_id,True
1,unique_product_id,True
2,unique_customer_id,True
3,unique_sales_id,True
4,sales_products_known,True
5,sales_customers_known,True
6,sales_regions_known,True
7,inventory_products_known,True
8,inventory_regions_known,True
9,crm_customers_known,True


,column,missing_after_conversion,minimum,maximum
0,products.base_list_price_eur,0,42.470,"1,719.990"
1,products.target_margin_pct,0,0.180,0.620
2,products.product_growth_factor,0,0.880,1.180
3,regions.market_growth_factor,0,0.950,1.140
4,sales.units,0,1.000,"1,890.000"
5,crm.activity_minutes,0,5.000,119.000
6,crm.customer_health_score,0,18.800,100.000
7,pipeline.weighted_pipeline_eur,0,28.590,"532,979.000"
8,pipeline.win_probability,0,0.003,0.975


## 6. Define relevant generation horizon

The date dimension extends beyond the observed business facts—for example, the supplied date sample reaches 2030 while sales end in 2025.

Generating market activity through the entire calendar would create unanchored future records and unnecessarily large files. The sales fact therefore defines the default operational horizon. A continuous list of months is created between the first and last sales month and the date dimension is checked for complete coverage.


In [10]:
sales_start_month = pd.Period(sales["year_month"].min(), freq="M")
sales_end_month = pd.Period(sales["year_month"].max(), freq="M")
months = (
    pd.period_range(sales_start_month, sales_end_month, freq="M")
    .astype(str)
    .tolist())

calendar_months = set(dates["year_month"].dropna())
missing_calendar_months = sorted(set(months) - calendar_months)
if missing_calendar_months:
    raise ValueError(
        "The date dimension does not cover all sales months: "
        f"{missing_calendar_months[:10]}")

region_ids = sorted(regions["region_id"].dropna().astype(int).unique())
product_ids = sorted(products["product_id"].dropna().astype(int).unique())
product_groups = sorted(products["product_group"].dropna().unique())

horizon_summary = pd.Series({
    "start_month": months[0],
    "end_month": months[-1],
    "months": len(months),
    "regions": len(region_ids),
    "products": len(product_ids),
    "product_groups": len(product_groups),
}, name="value")
display(horizon_summary.to_frame())


,value
start_month,2021-01
end_month,2025-12
months,60
regions,10
products,200
product_groups,10


## 7. Build product and customer calibration tables

In [11]:
products["decline_flag"] = (
    products["is_declining_product"].fillna(0).astype(int)
    if "is_declining_product" in products
    else products["lifecycle_stage"].str.casefold().eq("decline").astype(int))

product_info = products[[
    "product_id", "product_family", "product_group", "lifecycle_stage",
    "launch_date", "launch_month", "base_list_price_eur",
    "target_margin_pct", "product_growth_factor", "decline_flag",
]].copy()


first_sales_month = (
    sales.groupby("product_id", as_index=False)
    .agg(first_sales_month=("year_month", "min"))
)
product_info = product_info.merge(
    first_sales_month,
    on="product_id",
    how="left",
    validate="1:1")

configured_launch_date = pd.to_datetime(
    product_info["launch_month"].astype("string") + "-01",
    errors="coerce",
)
first_sales_date = pd.to_datetime(
    product_info["first_sales_month"].astype("string") + "-01",
    errors="coerce",
)
product_info["effective_launch_month"] = (
    pd.concat(
        [configured_launch_date, first_sales_date],
        axis=1,
    )
    .min(axis=1)
    .dt.strftime("%Y-%m")
)
product_info["launch_date_issue_flag"] = (
    configured_launch_date.notna()
    & first_sales_date.notna()
    & first_sales_date.lt(configured_launch_date)
).astype(int)
product_info["launch_month_gap"] = (
    (configured_launch_date.dt.year - first_sales_date.dt.year) * 12
    + configured_launch_date.dt.month
    - first_sales_date.dt.month
).where(product_info["launch_date_issue_flag"].eq(1), 0).fillna(0).astype(int)

sales_launch_audit = sales[[
    "sales_id", "product_id", "year_month",
]].merge(
    product_info[[
        "product_id", "launch_month", "first_sales_month",
        "effective_launch_month", "launch_date_issue_flag",
    ]],
    on="product_id",
    how="left",
    validate="m:1",
)
sales_launch_audit["before_configured_launch"] = (
    sales_launch_audit["launch_month"].notna()
    & (
        sales_launch_audit["year_month"]
        < sales_launch_audit["launch_month"]
    )
)
prelaunch_sales_by_product = (
    sales_launch_audit.loc[
        sales_launch_audit["before_configured_launch"]
    ]
    .groupby("product_id", as_index=False)
    .agg(
        sales_rows_before_configured_launch=("sales_id", "size"),
        earliest_sales_month=("year_month", "min"),
        latest_prelaunch_sales_month=("year_month", "max"),
    )
)
product_launch_issues = (
    product_info.loc[
        product_info["launch_date_issue_flag"].eq(1),
        [
            "product_id", "product_family", "product_group",
            "launch_month", "first_sales_month",
            "effective_launch_month", "launch_month_gap"]]
    .merge(
        prelaunch_sales_by_product,
        on="product_id",
        how="left",
        validate="1:1")
    .sort_values(
        ["sales_rows_before_configured_launch", "product_id"],
        ascending=[False, True]).reset_index(drop=True))

group_info = (
    product_info
    .groupby(
        ["product_family", "product_group"],
        as_index=False,
        dropna=False)
    .agg(
        product_count=("product_id", "nunique"),
        median_list_price_eur=("base_list_price_eur", "median"),
        average_target_margin_pct=("target_margin_pct", "mean"),
        average_product_growth_factor=(
            "product_growth_factor", "mean"),
        decline_product_share=("decline_flag", "mean")))

group_family_counts = group_info.groupby(
    "product_group"
)["product_family"].nunique()
ambiguous_groups = group_family_counts[group_family_counts > 1]
if not ambiguous_groups.empty:
    display(ambiguous_groups)
    raise ValueError(
        "product_group is not unique across product families. "
        "Pipeline data does not contain product_family, so the grouping "
        "key would be ambiguous.")

customer_region = customers[["customer_id", "region_id"]].copy()
display(group_info)


launch_quality_summary = pd.Series({
    "products": len(product_info),
    "products_with_sales": int(
        product_info["first_sales_month"].notna().sum()
    ),
    "products_with_launch_date_conflict": int(
        product_info["launch_date_issue_flag"].sum()
    ),
    "sales_rows_before_configured_launch": int(
        sales_launch_audit["before_configured_launch"].sum()
    ),
}, name="value")
print("Launch-date quality summary")
display(launch_quality_summary.to_frame())

if not product_launch_issues.empty:
    print(
        "Warning: configured launch dates conflict with observed sales. "
        "The effective launch month uses the earlier date so valid "
        "sales history remains covered."
    )
    display(product_launch_issues.head(10))


,product_family,product_group,product_count,median_list_price_eur,average_target_margin_pct,average_product_growth_factor,decline_product_share
0,Accessories,Accessory Kit,20,56.660,0.220,1.050,0.200
1,Components,Connectivity Module,20,83.355,0.310,1.180,0.200
2,IT Devices,Laptop Pro,20,911.995,0.290,1.080,0.150
3,IT Devices,Laptop Standard,20,606.045,0.240,1.010,0.200
4,IT Devices,Tablet Enterprise,20,413.980,0.270,1.120,0.150
5,Industrial Tech,Industrial Scanner,20,"1,443.040",0.340,1.030,0.050
6,Legacy Products,Legacy Workstation,20,705.930,0.180,0.880,0.800
7,Medical Devices,Medical Sensor,20,168.260,0.450,1.150,0.300
8,Medical Devices,Monitoring Device,20,798.415,0.390,1.100,0.250
9,Services,Service Contract,20,244.200,0.620,1.160,0.250


Launch-date quality summary


,value
products,200
products_with_sales,200
products_with_launch_date_conflict,101
sales_rows_before_configured_launch,19843


,product_id,product_family,product_group,launch_month,first_sales_month,effective_launch_month,launch_month_gap,sales_rows_before_configured_launch,earliest_sales_month,latest_prelaunch_sales_month
0,1077,Medical Devices,Medical Sensor,2024-01,2021-01,2021-01,36,461,2021-01,2023-12
1,1099,Medical Devices,Monitoring Device,2024-01,2021-01,2021-01,36,429,2021-01,2023-12
2,1054,IT Devices,Tablet Enterprise,2024-01,2021-01,2021-01,36,421,2021-01,2023-12
3,1007,IT Devices,Laptop Pro,2024-01,2021-01,2021-01,36,419,2021-01,2023-12
4,1097,Medical Devices,Monitoring Device,2024-01,2021-01,2021-01,36,418,2021-01,2023-12
5,1170,Accessories,Accessory Kit,2024-01,2021-01,2021-01,36,407,2021-01,2023-12
6,1171,Accessories,Accessory Kit,2024-01,2021-01,2021-01,36,399,2021-01,2023-12
7,1046,IT Devices,Tablet Enterprise,2024-01,2021-01,2021-01,36,391,2021-01,2023-12
8,1128,Components,Connectivity Module,2024-01,2021-01,2021-01,36,391,2021-01,2023-12
9,1092,Medical Devices,Monitoring Device,2024-01,2021-01,2021-01,36,381,2021-01,2023-12


## 8. Build the regional CRM driver

CRM activity is aggregated by month and customer region. A complete month–region panel ensures that regions with no recorded activities are represented explicitly.

`regional_crm_activity_index` compares each region with the average CRM activity in the same month:

- 100 = monthly regional average,
- above 100 = more activity than average,
- 0 = no activity when other regions were active,
- 100 = neutral when no region had any activity that month.

In [12]:
crm_enriched = (
    crm.drop(columns=["region_id"], errors="ignore")
    .merge(
        customer_region,
        on="customer_id",
        how="left",
        validate="m:1"))

if crm_enriched["region_id"].isna().any():
    missing_customers = (
        crm_enriched.loc[
            crm_enriched["region_id"].isna(), "customer_id"]
        .drop_duplicates()
        .head(10)
        .tolist())
    raise ValueError(
        f"CRM rows contain unknown customers: {missing_customers}")

crm_observed = (
    crm_enriched.loc[crm_enriched["year_month"].isin(months)]
    .groupby(["year_month", "region_id"], as_index=False)
    .agg(
        crm_activity_count=("activity_id", "nunique"),
        crm_activity_minutes=("activity_minutes", "sum"),
        average_customer_health=("customer_health_score", "mean")))

region_month_panel = pd.MultiIndex.from_product(
    [months, region_ids],
    names=["year_month", "region_id"],
).to_frame(index=False)

crm_region_month = region_month_panel.merge(
    crm_observed,
    on=["year_month", "region_id"],
    how="left",
    validate="1:1")
crm_region_month[[
    "crm_activity_count", "crm_activity_minutes",
]] = crm_region_month[[
    "crm_activity_count", "crm_activity_minutes",
]].fillna(0)
crm_region_month["average_customer_health"] = crm_region_month[
    "average_customer_health"
].fillna(65)

monthly_crm_average = crm_region_month.groupby(
    "year_month"
)["crm_activity_count"].transform("mean")
crm_index = (
    100
    * crm_region_month["crm_activity_count"]
    / monthly_crm_average.replace(0, np.nan))
crm_index = crm_index.where(monthly_crm_average.ne(0), 100)
crm_region_month["regional_crm_activity_index"] = (
    crm_index.fillna(0).clip(0, 250).round(2))


In [13]:
# Check CRM driver
crm_driver_summary = crm_region_month[[
    "crm_activity_count", "crm_activity_minutes",
    "average_customer_health", "regional_crm_activity_index",
]].describe().T
display(crm_driver_summary)

print("Example month across regions")
display(
    crm_region_month.loc[
        crm_region_month["year_month"].eq(months[0])].sort_values("region_id"))


,count,mean,std,min,25%,50%,75%,max
crm_activity_count,600.000,58.333,27.078,18.000,38.000,53.000,66.250,141.000
crm_activity_minutes,600.000,"2,041.912",961.005,577.000,"1,360.500","1,850.000","2,372.000","5,038.000"
average_customer_health,600.000,66.268,1.641,61.206,65.273,66.300,67.324,71.748
regional_crm_activity_index,600.000,100.000,46.087,30.100,65.492,92.345,113.225,235.400


Example month across regions


,year_month,region_id,crm_activity_count,crm_activity_minutes,average_customer_health,regional_crm_activity_index
0,2021-01,1,122,4376,65.890,194.890
1,2021-01,2,67,2542,67.984,107.030
2,2021-01,3,62,2090,65.345,99.040
3,2021-01,4,51,1837,67.400,81.470
4,2021-01,5,66,2357,66.130,105.430
5,2021-01,6,100,3479,64.839,159.740
6,2021-01,7,36,1071,67.756,57.510
7,2021-01,8,54,2056,66.337,86.260
8,2021-01,9,29,912,68.810,46.330
9,2021-01,10,39,1288,67.674,62.300


## 9. Build the product-group pipeline driver

Pipeline opportunities are assigned to regions through the customer dimension and to canonical product groups through the product dimension.

The resulting `pipeline_interest_index` compares opportunity volume for each product group and region with the overall combination average in the same month.


In [14]:
pipeline_enriched = (
    pipeline.drop(columns=["region_id"], errors="ignore")
    .merge(
        customer_region,
        on="customer_id",
        how="left",
        validate="m:1"))

if pipeline_enriched["region_id"].isna().any():
    unknown_customers = (
        pipeline_enriched.loc[
            pipeline_enriched["region_id"].isna(), "customer_id"
        ]
        .drop_duplicates()
        .head(10)
        .tolist())
    raise ValueError(
        f"Pipeline rows contain unknown customers: {unknown_customers}")

pipeline_observed = (
    pipeline_enriched.loc[
        pipeline_enriched["year_month"].isin(months)]
    .groupby(
        ["year_month", "product_group", "region_id"],
        as_index=False)
    .agg(
        opportunity_count=("opportunity_id", "nunique"),
        weighted_pipeline_eur=("weighted_pipeline_eur", "sum"),
        average_win_probability=("win_probability", "mean")))

signal_panel = pd.MultiIndex.from_product(
    [months, product_groups, region_ids],
    names=["year_month", "product_group", "region_id"],
).to_frame(index=False)

pipeline_group_month = signal_panel.merge(
    pipeline_observed,
    on=["year_month", "product_group", "region_id"],
    how="left",
    validate="1:1")
pipeline_group_month[[
    "opportunity_count", "weighted_pipeline_eur",
]] = pipeline_group_month[[
    "opportunity_count", "weighted_pipeline_eur",
]].fillna(0)
pipeline_group_month["average_win_probability"] = (
    pipeline_group_month["average_win_probability"].fillna(0))

monthly_pipeline_average = pipeline_group_month.groupby(
    "year_month"
)["opportunity_count"].transform("mean")
pipeline_group_month["pipeline_interest_index"] = (
    100
    * pipeline_group_month["opportunity_count"]
    / monthly_pipeline_average.replace(0, np.nan)
).fillna(0).clip(0, 300).round(2)


In [15]:
# Check pipeline driver
pipeline_driver_summary = pipeline_group_month[[
    "opportunity_count", "weighted_pipeline_eur",
    "average_win_probability", "pipeline_interest_index",
]].describe().T
display(pipeline_driver_summary)

zero_pipeline_share = pipeline_group_month[
    "opportunity_count"
].eq(0).mean()
print(f"Rows with no pipeline opportunities: {zero_pipeline_share:.1%}")

,count,mean,std,min,25%,50%,75%,max
opportunity_count,"6,000.000",2.500,1.923,0.000,1.000,2.000,4.000,12.000
weighted_pipeline_eur,"6,000.000","46,148.674","55,602.152",0.000,"8,691.502","28,988.025","63,786.495","781,059.080"
average_win_probability,"6,000.000",0.371,0.192,0.000,0.276,0.398,0.493,0.930
pipeline_interest_index,"6,000.000",98.989,73.339,0.000,40.650,82.300,139.493,300.000


Rows with no pipeline opportunities: 12.3%


## 10. Build the product-group supply driver

Inventory is aggregated to the same grain as market signals. Supply pressure increases when:
- the share of stockout products is high or
- ending inventory is low relative to opening stock plus production.

Missing inventory combinations receive a neutral pressure value rather than being interpreted as a stockout.

In [16]:
inventory_enriched = inventory.merge(
    product_info[["product_id", "product_group"]],
    on="product_id",
    how="left",
    validate="m:1")

inventory_enriched["stockout_flag_numeric"] = (
    inventory_enriched["stockout_flag"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .isin(["true", "1", "yes"])
    .astype(int))
inventory_enriched["available_stock_ratio"] = (
    inventory_enriched["ending_stock_units"]
    / (
        inventory_enriched["opening_stock_units"]
        + inventory_enriched["production_units"]
    ).replace(0, np.nan)
).clip(0, 1)

inventory_observed = (
    inventory_enriched.loc[
        inventory_enriched["year_month"].isin(months)
    ]
    .groupby(
        ["year_month", "product_group", "region_id"],
        as_index=False,
    )
    .agg(
        stockout_rate=("stockout_flag_numeric", "mean"),
        average_available_stock_ratio=(
            "available_stock_ratio", "mean")))

inventory_group_month = signal_panel.merge(
    inventory_observed,
    on=["year_month", "product_group", "region_id"],
    how="left",
    validate="1:1")
inventory_group_month["inventory_observed_flag"] = (
    inventory_group_month["stockout_rate"].notna().astype(int))
inventory_group_month["stockout_rate"] = (
    inventory_group_month["stockout_rate"].fillna(0))
inventory_group_month["average_available_stock_ratio"] = (
    inventory_group_month["average_available_stock_ratio"].fillna(0.50))
inventory_group_month["supply_pressure_index"] = (
    25
    + 65 * inventory_group_month["stockout_rate"]
    + 35 * (1 - inventory_group_month["average_available_stock_ratio"])
).clip(5, 100).round(2)


In [17]:
# Check supply driver
supply_driver_summary = inventory_group_month[[
    "stockout_rate", "average_available_stock_ratio",
    "supply_pressure_index", "inventory_observed_flag",
]].describe().T
display(supply_driver_summary)

inventory_coverage = inventory_group_month[
    "inventory_observed_flag"
].mean()
print(f"Signal combinations with observed inventory: {inventory_coverage:.1%}")


,count,mean,std,min,25%,50%,75%,max
stockout_rate,"6,000.000",0.003,0.018,0.000,0.000,0.000,0.000,0.200
average_available_stock_ratio,"6,000.000",0.541,0.029,0.316,0.523,0.542,0.560,0.654
supply_pressure_index,"6,000.000",41.288,1.752,37.120,40.400,41.040,41.710,58.600
inventory_observed_flag,"6,000.000",1.000,0.000,1.000,1.000,1.000,1.000,1.000


Signal combinations with observed inventory: 100.0%


## 11. Build a leakage-safe historical sales calibration driver

Internal sales should not be disguised as external market demand. They are therefore used only as a small, lagged calibration component:

- previous-month product-group units are compared with an earlier six-month baseline;
- current-month and future sales are never used;
- the driver is internal and is not exported.

This keeps simulated demand aligned with the project's commercial history without copying the prediction target.

In [18]:
sales_enriched = sales.merge(
    product_info[["product_id", "product_group"]],
    on="product_id",
    how="left",
    validate="m:1")
sales_observed = (
    sales_enriched.loc[sales_enriched["year_month"].isin(months)]
    .groupby(
        ["year_month", "product_group", "region_id"],
        as_index=False)
    .agg(observed_units=("units", "sum")))

sales_group_month = signal_panel.merge(
    sales_observed,
    on=["year_month", "product_group", "region_id"],
    how="left",
    validate="1:1",
).fillna({"observed_units": 0})
sales_group_month = sales_group_month.sort_values(
    ["product_group", "region_id", "year_month"]
).reset_index(drop=True)
sales_history = sales_group_month.groupby(
    ["product_group", "region_id"],
    group_keys=False,
)["observed_units"]
sales_group_month["units_lag_1m"] = sales_history.shift(1)
sales_group_month["prior_units_baseline_6m"] = sales_history.transform(
    lambda series: series.shift(2).rolling(6, min_periods=2).mean())
sales_group_month["historical_sales_momentum_index"] = (
    100
    * sales_group_month["units_lag_1m"]
    / sales_group_month["prior_units_baseline_6m"].replace(0, np.nan)
).fillna(100).clip(0, 250)


In [19]:
# Check lagged sales calibration
example_sales_history = (
    sales_group_month.loc[
        (sales_group_month["product_group"] == product_groups[0])
        & (sales_group_month["region_id"] == region_ids[0]),
        [
            "year_month", "observed_units", "units_lag_1m",
            "prior_units_baseline_6m",
            "historical_sales_momentum_index"]]
    .head(12))
display(example_sales_history)


,year_month,observed_units,units_lag_1m,prior_units_baseline_6m,historical_sales_momentum_index
0,2021-01,1413,<NA>,NaN,100.000
1,2021-02,1287,1413,NaN,100.000
2,2021-03,2024,1287,NaN,100.000
3,2021-04,1612,2024,"1,350.000",149.926
4,2021-05,1879,1612,"1,574.667",102.371
5,2021-06,1438,1879,"1,584.000",118.624
6,2021-07,1060,1438,"1,643.000",87.523
7,2021-08,949,1060,"1,608.833",65.886
8,2021-09,1101,949,"1,550.000",61.226
9,2021-10,1495,1101,"1,493.667",73.711


## 12. Define the market-signal generation assumptions

The formula combines structural and operational components:

- source seasonality: Q4 uplift and summer slowdown from the base sales generator
- a smooth macroeconomic index with small random variation
- regional and product-group growth factors
- lifecycle decline exposure
- occasional positive or negative demand shocks
- a small weight for lagged internal sales momentum

Competition follows a mean-reverting process so that it changes gradually rather than jumping randomly every month.

In [20]:
SIGNAL_ASSUMPTIONS = {
    "q4_seasonality_index": 130.0,
    "summer_seasonality_index": 82.0,
    "default_seasonality_index": 100.0,
    "monthly_demand_shock_probability": 0.032,
    "negative_shock_probability": 0.68,
    "lagged_sales_calibration_weight": 0.12,
    "structural_demand_weight": 0.88,
    "market_demand_min": 42.0,
    "market_demand_max": 180.0}
display(
    pd.Series(SIGNAL_ASSUMPTIONS, name="value")
    .rename_axis("assumption")
    .reset_index())

,assumption,value
0,q4_seasonality_index,130.000
1,summer_seasonality_index,82.000
2,default_seasonality_index,100.000
3,monthly_demand_shock_probability,0.032
4,negative_shock_probability,0.680
5,lagged_sales_calibration_weight,0.120
6,structural_demand_weight,0.880
7,market_demand_min,42.000
8,market_demand_max,180.000


In [21]:
# Return the seasonal pattern used by the base sales generator
def source_seasonality_index(year_month: str) -> float:
    month_number = int(year_month[-2:])
    if month_number in {10, 11, 12}:
        return SIGNAL_ASSUMPTIONS["q4_seasonality_index"]
    if month_number in {7, 8}:
        return SIGNAL_ASSUMPTIONS["summer_seasonality_index"]
    return SIGNAL_ASSUMPTIONS["default_seasonality_index"]

# Generate reproducible market signals at month × group × region grain
def create_market_signals(
    groups: pd.DataFrame,
    region_dimension: pd.DataFrame,
    month_values: list[str],
    pipeline_driver: pd.DataFrame,
    supply_driver: pd.DataFrame,
    sales_driver: pd.DataFrame,
    rng: np.random.Generator,
) -> pd.DataFrame:
    pipeline_lookup = pipeline_driver.set_index(
        ["year_month", "product_group", "region_id"]
    )["pipeline_interest_index"]
    supply_lookup = supply_driver.set_index(
        ["year_month", "product_group", "region_id"]
    )["supply_pressure_index"]
    sales_lookup = sales_driver.set_index(
        ["year_month", "product_group", "region_id"]
    )["historical_sales_momentum_index"]
    region_lookup = region_dimension.set_index("region_id").to_dict("index")

    macro_by_month = {}
    for month_index, year_month in enumerate(month_values):
        macro_by_month[year_month] = float(np.clip(
            100
            + 0.14 * month_index
            + 3.5 * np.sin(month_index / 6.5)
            + rng.normal(0, 1.1),
            88,
            120))

    rows = []
    for _, group in groups.sort_values("product_group").iterrows():
        for region_id in sorted(region_lookup):
            region = region_lookup[region_id]
            competitor_state = rng.uniform(35, 74)

            for month_index, year_month in enumerate(month_values):
                seasonality_index = source_seasonality_index(year_month)
                macro_index = macro_by_month[year_month]
                years_elapsed = month_index / 12

                competitor_state = np.clip(
                    0.84 * competitor_state
                    + 0.16 * 55
                    + rng.normal(0, 3.8),
                    10,
                    95)

                key = (year_month, group["product_group"], region_id)
                pipeline_index = float(pipeline_lookup.loc[key])
                supply_pressure = float(supply_lookup.loc[key])
                lagged_sales_index = float(sales_lookup.loc[key])

                shock_flag = int(
                    rng.random()
                    < SIGNAL_ASSUMPTIONS[
                        "monthly_demand_shock_probability"])
                if shock_flag:
                    negative_shock = (
                        rng.random()
                        < SIGNAL_ASSUMPTIONS[
                            "negative_shock_probability"])
                    shock_multiplier = (
                        rng.uniform(0.76, 0.90)
                        if negative_shock
                        else rng.uniform(1.10, 1.24))
                else:
                    shock_multiplier = 1.0

                region_growth = (
                    float(region["market_growth_factor"])
                    ** years_elapsed)
                product_growth = (
                    float(group["average_product_growth_factor"])** years_elapsed)
                lifecycle_factor = max(
                    0.72,
                    1
                    - 0.18
                    * float(group["decline_product_share"])
                    * years_elapsed)

                structural_demand = (
                    100
                    * seasonality_index / 100
                    * macro_index / 100
                    * region_growth
                    * product_growth
                    * lifecycle_factor
                    * shock_multiplier
                    * rng.lognormal(0, 0.035))
                demand_index = (
                    SIGNAL_ASSUMPTIONS["structural_demand_weight"]
                    * structural_demand
                    + SIGNAL_ASSUMPTIONS[
                        "lagged_sales_calibration_weight"]
                    * lagged_sales_index)
                demand_index = float(np.clip(
                    demand_index,
                    SIGNAL_ASSUMPTIONS["market_demand_min"],
                    SIGNAL_ASSUMPTIONS["market_demand_max"]))

                opportunity_score = float(np.clip(
                    0.40 * demand_index
                    + 0.22 * (100 - competitor_state)
                    + 0.15 * (100 - supply_pressure)
                    + 0.23 * min(pipeline_index, 100),
                    0,
                    100))

                rows.append({
                    "year_month": year_month,
                    "product_family": group["product_family"],
                    "product_group": group["product_group"],
                    "region_id": int(region_id),
                    "market_demand_index": round(demand_index, 2),
                    "competitor_pressure_index": round(
                        float(competitor_state), 2),
                    "seasonality_index": round(seasonality_index, 2),
                    "macro_business_index": round(macro_index, 2),
                    "supply_pressure_index": round(supply_pressure, 2),
                    "pipeline_interest_index": round(pipeline_index, 2),
                    "demand_shock_flag": shock_flag,
                    "market_opportunity_score": round(
                        opportunity_score, 2),
                    "regional_market_growth_factor": round(
                        float(region["market_growth_factor"]), 3)})

    output = (
        pd.DataFrame(rows)
        .sort_values(["product_group", "region_id", "year_month"])
        .reset_index(drop=True))
    output["market_growth_pct"] = (
        output.groupby(
            ["product_group", "region_id"]
        )["market_demand_index"]
        .pct_change(fill_method=None)
        .mul(100)
        .fillna(0)
        .clip(-50, 50)
        .round(2))

    final_columns = [
        "year_month", "product_family", "product_group", "region_id",
        "market_demand_index", "market_growth_pct",
        "competitor_pressure_index", "seasonality_index",
        "macro_business_index", "supply_pressure_index",
        "pipeline_interest_index", "demand_shock_flag",
        "market_opportunity_score",
        "regional_market_growth_factor"]
    return output[final_columns]


## 13. Generate market signals

In [22]:
signal_rng = np.random.default_rng(MARKET_SIGNAL_SEED)
market_signals = create_market_signals(
    groups=group_info,
    region_dimension=regions,
    month_values=months,
    pipeline_driver=pipeline_group_month,
    supply_driver=inventory_group_month,
    sales_driver=sales_group_month,
    rng=signal_rng)

print(
    f"market_signals: {len(market_signals):,} rows × "
    f"{market_signals.shape[1]} columns")
display(market_signals.head())


market_signals: 6,000 rows × 14 columns


,year_month,product_family,product_group,region_id,market_demand_index,market_growth_pct,competitor_pressure_index,seasonality_index,macro_business_index,supply_pressure_index,pipeline_interest_index,demand_shock_flag,market_opportunity_score,regional_market_growth_factor
0,2021-01,Accessories,Accessory Kit,1,102.070,0.000,71.060,100.000,100.260,41.440,278.880,0,78.980,1.060
1,2021-02,Accessories,Accessory Kit,1,98.970,-3.040,61.980,100.000,103.460,41.730,134.530,0,79.690,1.060
2,2021-03,Accessories,Accessory Kit,1,99.520,0.560,62.130,100.000,101.860,41.180,181.820,0,79.960,1.060
3,2021-04,Accessories,Accessory Kit,1,111.100,11.640,64.880,100.000,102.810,41.230,80.970,0,79.600,1.060
4,2021-05,Accessories,Accessory Kit,1,102.280,-7.940,65.650,100.000,102.070,40.890,113.210,0,80.340,1.060


In [23]:
# Check signal distributions and one history
signal_distribution = market_signals[[
    "market_demand_index", "market_growth_pct",
    "competitor_pressure_index", "macro_business_index",
    "supply_pressure_index", "pipeline_interest_index",
    "market_opportunity_score"]].describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]).T
display(signal_distribution)

demand_at_lower_bound = market_signals["market_demand_index"].eq(
    SIGNAL_ASSUMPTIONS["market_demand_min"]
).mean()
demand_at_upper_bound = market_signals["market_demand_index"].eq(
    SIGNAL_ASSUMPTIONS["market_demand_max"]
).mean()
print(f"Demand at lower bound: {demand_at_lower_bound:.2%}")
print(f"Demand at upper bound: {demand_at_upper_bound:.2%}")

example_signal_history = market_signals.loc[
    (market_signals["product_group"] == product_groups[0])
    & (market_signals["region_id"] == region_ids[0])
].head(12)
display(example_signal_history)

,count,mean,std,min,1%,25%,50%,75%,99%,max
market_demand_index,"6,000.000",127.244,34.480,42.000,51.140,102.178,122.560,155.055,180.000,180.000
market_growth_pct,"6,000.000",1.680,14.020,-46.210,-28.300,-5.825,0.000,7.732,41.531,50.000
competitor_pressure_index,"6,000.000",55.245,7.403,25.560,37.840,50.308,55.310,60.190,72.241,81.750
macro_business_index,"6,000.000",104.841,3.642,99.180,99.180,102.030,103.905,108.410,112.250,112.250
supply_pressure_index,"6,000.000",41.288,1.752,37.120,38.910,40.400,41.040,41.710,49.661,58.600
pipeline_interest_index,"6,000.000",98.989,73.339,0.000,0.000,40.650,82.300,139.493,300.000,300.000
market_opportunity_score,"6,000.000",83.788,13.891,34.980,48.120,74.312,85.110,97.775,100.000,100.000


Demand at lower bound: 0.25%
Demand at upper bound: 14.50%


,year_month,product_family,product_group,region_id,market_demand_index,market_growth_pct,competitor_pressure_index,seasonality_index,macro_business_index,supply_pressure_index,pipeline_interest_index,demand_shock_flag,market_opportunity_score,regional_market_growth_factor
0,2021-01,Accessories,Accessory Kit,1,102.070,0.000,71.060,100.000,100.260,41.440,278.880,0,78.980,1.060
1,2021-02,Accessories,Accessory Kit,1,98.970,-3.040,61.980,100.000,103.460,41.730,134.530,0,79.690,1.060
2,2021-03,Accessories,Accessory Kit,1,99.520,0.560,62.130,100.000,101.860,41.180,181.820,0,79.960,1.060
3,2021-04,Accessories,Accessory Kit,1,111.100,11.640,64.880,100.000,102.810,41.230,80.970,0,79.600,1.060
4,2021-05,Accessories,Accessory Kit,1,102.280,-7.940,65.650,100.000,102.070,40.890,113.210,0,80.340,1.060
5,2021-06,Accessories,Accessory Kit,1,107.340,4.950,66.190,100.000,102.970,40.890,170.210,0,82.240,1.060
6,2021-07,Accessories,Accessory Kit,1,84.350,-21.420,59.600,82.000,103.970,40.720,122.450,0,74.520,1.060
7,2021-08,Accessories,Accessory Kit,1,86.750,2.850,55.980,82.000,103.000,40.690,202.430,0,76.280,1.060
8,2021-09,Accessories,Accessory Kit,1,103.730,19.570,51.460,100.000,104.280,41.300,130.430,0,83.980,1.060
9,2021-10,Accessories,Accessory Kit,1,135.960,31.070,50.570,130.000,103.280,44.510,264.150,0,96.580,1.060


## 14. Build the active product–region–month panel

each product becomes eligible in its launch month. This prevents impossible pre-launch campaigns and makes the expected row count transparent.

In [24]:
eligibility_activity_panel = (
    product_info.assign(_join_key=1)
    .merge(
        pd.DataFrame({"year_month": months, "_join_key": 1}),
        on="_join_key",
        how="inner",
    )
    .drop(columns="_join_key")
)
eligibility_activity_panel = eligibility_activity_panel.loc[
    eligibility_activity_panel["effective_launch_month"].isna()
    | (
        eligibility_activity_panel["year_month"]
        >= eligibility_activity_panel["effective_launch_month"]
    )
].copy()
eligibility_activity_panel = (
    eligibility_activity_panel.assign(_join_key=1)
    .merge(
        pd.DataFrame({"region_id": region_ids, "_join_key": 1}),
        on="_join_key",
        how="inner",
    )
    .drop(columns="_join_key")
    .sort_values(["product_id", "region_id", "year_month"])
    .reset_index(drop=True)
)

activity_key = ["year_month", "product_id", "region_id"]
observed_sales_activity_rows = (
    sales[activity_key]
    .drop_duplicates()
    .merge(
        product_info,
        on="product_id",
        how="left",
        validate="m:1",
    )
)

sales_keys_missing_from_eligibility = (
    observed_sales_activity_rows[activity_key]
    .merge(
        eligibility_activity_panel[activity_key],
        on=activity_key,
        how="left",
        indicator=True,
        validate="1:1",
    )
    .loc[lambda frame: frame["_merge"].eq("left_only"), activity_key]
    .reset_index(drop=True)
)

# Sales keys are appended last and retained when a key already exists.
activity_panel = (
    pd.concat(
        [eligibility_activity_panel, observed_sales_activity_rows],
        ignore_index=True,
    )
    .drop_duplicates(activity_key, keep="last")
    .sort_values(["product_id", "region_id", "year_month"])
    .reset_index(drop=True)
)

pre_launch_rows = (
    activity_panel["effective_launch_month"].notna()
    & (
        activity_panel["year_month"]
        < activity_panel["effective_launch_month"]
    )
).sum()
print(f"Eligible product–region–month rows: {len(activity_panel):,}")
print(f"Rows before effective launch retained: {pre_launch_rows:,}")
print(
    "Observed sales keys added by coverage safeguard: "
    f"{len(sales_keys_missing_from_eligibility):,}"
)
if not sales_keys_missing_from_eligibility.empty:
    display(sales_keys_missing_from_eligibility.head(10))
display(activity_panel.head())


Eligible product–region–month rows: 120,000
Rows before effective launch retained: 0
Observed sales keys added by coverage safeguard: 0


,product_id,product_family,product_group,lifecycle_stage,launch_date,launch_month,base_list_price_eur,target_margin_pct,product_growth_factor,decline_flag,first_sales_month,effective_launch_month,launch_date_issue_flag,launch_month_gap,year_month,region_id
0,1000,IT Devices,Laptop Pro,Decline,2022-01-01,2022-01,804.370,0.290,1.080,1,2021-01,2021-01,1,12,2021-01,1
1,1000,IT Devices,Laptop Pro,Decline,2022-01-01,2022-01,804.370,0.290,1.080,1,2021-01,2021-01,1,12,2021-02,1
2,1000,IT Devices,Laptop Pro,Decline,2022-01-01,2022-01,804.370,0.290,1.080,1,2021-01,2021-01,1,12,2021-03,1
3,1000,IT Devices,Laptop Pro,Decline,2022-01-01,2022-01,804.370,0.290,1.080,1,2021-01,2021-01,1,12,2021-04,1
4,1000,IT Devices,Laptop Pro,Decline,2022-01-01,2022-01,804.370,0.290,1.080,1,2021-01,2021-01,1,12,2021-05,1


## 15. Define marketing-activity assumptions

Campaign probability increases for:

- higher-margin products
- high-consideration product families
- new and growth-stage products
- strong market opportunities
- above-average CRM activity
- strong pipeline interest
- and demand-shock months

Website visits include both organic interest and campaign clicks. Demo and lead probabilities vary by product characteristics and campaign status.

`cost_per_lead_eur` is intentionally missing when there is no paid spend or no lead. Reporting zero would incorrectly suggest perfect efficiency.

In [25]:
ACTIVITY_ASSUMPTIONS = {
    "campaign_probability_min": 0.04,
    "campaign_probability_max": 0.36,
    "channels": [
        "Paid Search", "Email", "Partner",
        "Trade Fair", "Webinar", "Content"],
    "channel_probabilities": [0.23, 0.23, 0.18, 0.10, 0.13, 0.13],
    "high_consideration_families": [
        "Medical Devices", "Industrial Tech", "Services"]}
display(pd.Series({
    "campaign_probability_min": ACTIVITY_ASSUMPTIONS[
        "campaign_probability_min"],
    "campaign_probability_max": ACTIVITY_ASSUMPTIONS[
        "campaign_probability_max"],
    "channels": ", ".join(ACTIVITY_ASSUMPTIONS["channels"]),
    "high_consideration_families": ", ".join(
        ACTIVITY_ASSUMPTIONS["high_consideration_families"]),
}, name="value").rename_axis("assumption").reset_index())

,assumption,value
0,campaign_probability_min,0.040
1,campaign_probability_max,0.360
2,channels,"Paid Search, Email, Partner, Trade Fair, Webin..."
3,high_consideration_families,"Medical Devices, Industrial Tech, Services"


In [26]:
def create_market_activity(
    active_panel: pd.DataFrame,
    signals: pd.DataFrame,
    crm_driver: pd.DataFrame,
    rng: np.random.Generator,
) -> pd.DataFrame:
    '''Generate reproducible activity at active month × product × region grain.'''
    signal_lookup = signals.set_index(
        ["year_month", "product_group", "region_id"]
    )
    crm_lookup = crm_driver.set_index(
        ["year_month", "region_id"]
    )["regional_crm_activity_index"]

    products = active_panel.copy()
    price_log = np.log1p(products["base_list_price_eur"].astype(float))
    price_range = price_log.max() - price_log.min()
    products["_price_score"] = (
        (price_log - price_log.min())
        / max(float(price_range), 1e-9)
    )
    products["_margin_score"] = products[
        "target_margin_pct"
    ].astype(float).clip(0, 1)

    channels = ACTIVITY_ASSUMPTIONS["channels"]
    channel_probabilities = ACTIVITY_ASSUMPTIONS[
        "channel_probabilities"
    ]
    high_consideration_families = {
        value.casefold()
        for value in ACTIVITY_ASSUMPTIONS[
            "high_consideration_families"
        ]
    }

    rows = []
    for _, product in products.iterrows():
        signal_key = (
            product["year_month"],
            product["product_group"],
            int(product["region_id"]),
        )
        signal = signal_lookup.loc[signal_key]
        crm_index = float(crm_lookup.loc[(
            product["year_month"],
            int(product["region_id"]),
        )])
        pipeline_index = float(signal["pipeline_interest_index"])

        high_consideration = (
            str(product["product_family"]).casefold()
            in high_consideration_families
        )
        launch_or_growth = (
            str(product["lifecycle_stage"]).casefold()
            in {"new", "growth"}
        )

        campaign_probability = float(np.clip(
            0.045
            + 0.070 * product["_margin_score"]
            + 0.060 * high_consideration
            + 0.055 * launch_or_growth
            + 0.045 * (
                signal["market_opportunity_score"] >= 58
            )
            + 0.035 * (crm_index >= 105)
            + 0.030 * (pipeline_index >= 110)
            + 0.020 * signal["demand_shock_flag"],
            ACTIVITY_ASSUMPTIONS["campaign_probability_min"],
            ACTIVITY_ASSUMPTIONS["campaign_probability_max"],
        ))
        campaign_flag = int(rng.random() < campaign_probability)

        if campaign_flag:
            campaign_channel = str(
                rng.choice(channels, p=channel_probabilities)
            )
            spend = (
                rng.gamma(3.0, 820.0)
                * (0.80 + 1.15 * product["_price_score"])
                * (0.75 + crm_index / 280)
            )
            impressions = int(max(
                100,
                rng.normal(
                    spend * rng.uniform(13, 29),
                    max(spend * 2.4, 1),
                ),
            ))
            expected_ctr = float(np.clip(
                0.017
                + 0.011
                * signal["market_opportunity_score"] / 100
                - 0.004
                * signal["competitor_pressure_index"] / 100
                + rng.normal(0, 0.002),
                0.006,
                0.055,
            ))
            clicks = int(rng.binomial(impressions, expected_ctr))
        else:
            campaign_channel = "No Active Campaign"
            spend = 0.0
            impressions = 0
            clicks = 0

        organic_visits = (
            28
            + 120 * product["_price_score"]
            + 1.10 * signal["market_demand_index"]
            + 0.18 * min(pipeline_index, 200)
            + rng.normal(0, 25)
        )
        website_visits = int(max(
            0,
            organic_visits + clicks * rng.uniform(0.80, 1.20),
        ))
        product_page_views = int(max(
            0,
            website_visits * rng.uniform(1.25, 2.80),
        ))

        demo_rate = float(np.clip(
            0.003
            + 0.007 * high_consideration
            + 0.003 * launch_or_growth
            + 0.002 * campaign_flag,
            0.002,
            0.020,
        ))
        demo_requests = int(
            rng.binomial(product_page_views, demo_rate)
        )
        click_leads = (
            int(rng.binomial(clicks, 0.055))
            if clicks > 0
            else 0
        )
        organic_leads = int(
            rng.binomial(website_visits, 0.002)
        )
        marketing_qualified_leads = (
            demo_requests + click_leads + organic_leads
        )

        campaign_ctr_pct = (
            100 * clicks / impressions
            if impressions > 0
            else 0.0
        )
        cost_per_lead = (
            spend / marketing_qualified_leads
            if spend > 0 and marketing_qualified_leads > 0
            else np.nan
        )

        rows.append({
            "year_month": product["year_month"],
            "product_id": int(product["product_id"]),
            "region_id": int(product["region_id"]),
            "campaign_flag": campaign_flag,
            "campaign_channel": campaign_channel,
            "campaign_spend_eur": round(float(spend), 2),
            "campaign_impressions": impressions,
            "campaign_clicks": clicks,
            "campaign_ctr_pct": round(
                float(campaign_ctr_pct), 3),
            "website_visits": website_visits,
            "product_page_views": product_page_views,
            "demo_requests": demo_requests,
            "marketing_qualified_leads": marketing_qualified_leads,
            "cost_per_lead_eur": (
                round(float(cost_per_lead), 2)
                if np.isfinite(cost_per_lead)
                else np.nan),
            "regional_crm_activity_index": round(crm_index, 2),
            "pipeline_interest_index": round(pipeline_index, 2)})

    return (
        pd.DataFrame(rows)
        .sort_values(["product_id", "region_id", "year_month"])
        .reset_index(drop=True)
    )


## 16. Generate market activity

In [27]:
activity_rng = np.random.default_rng(MARKET_ACTIVITY_SEED)
market_activity = create_market_activity(
    active_panel=activity_panel,
    signals=market_signals,
    crm_driver=crm_region_month,
    rng=activity_rng)

print(
    f"market_activity: {len(market_activity):,} rows × "
    f"{market_activity.shape[1]} columns")
display(market_activity.head())


market_activity: 120,000 rows × 16 columns


,year_month,product_id,region_id,campaign_flag,campaign_channel,campaign_spend_eur,campaign_impressions,campaign_clicks,campaign_ctr_pct,website_visits,product_page_views,demo_requests,marketing_qualified_leads,cost_per_lead_eur,regional_crm_activity_index,pipeline_interest_index
0,2021-01,1000,1,0,No Active Campaign,0.000,0,0,0.000,270,514,1,1,NaN,194.890,119.520
1,2021-02,1000,1,0,No Active Campaign,0.000,0,0,0.000,264,354,2,2,NaN,205.660,179.370
2,2021-03,1000,1,1,Trade Fair,"6,135.830",119734,3129,2.613,3137,8698,36,225,27.270,207.360,109.090
3,2021-04,1000,1,0,No Active Campaign,0.000,0,0,0.000,242,432,4,4,NaN,198.970,80.970
4,2021-05,1000,1,1,Content,"3,101.770",63840,1621,2.539,1634,4185,20,104,29.820,175.790,188.680


In [28]:
# Check activity distribution and campaign behaviour
campaign_summary = (
    market_activity.groupby("campaign_flag")
    .agg(
        rows=("product_id", "size"),
        average_spend_eur=("campaign_spend_eur", "mean"),
        average_impressions=("campaign_impressions", "mean"),
        average_website_visits=("website_visits", "mean"),
        average_demo_requests=("demo_requests", "mean"),
        average_mqls=("marketing_qualified_leads", "mean"),
        median_cost_per_lead_eur=("cost_per_lead_eur", "median")).round(2))
display(campaign_summary)

channel_summary = (
    market_activity.loc[market_activity["campaign_flag"].eq(1)]
    .groupby("campaign_channel")
    .agg(
        campaigns=("product_id", "size"),
        average_spend_eur=("campaign_spend_eur", "mean"),
        average_ctr_pct=("campaign_ctr_pct", "mean"),
        average_mqls=("marketing_qualified_leads", "mean"))
    .sort_values("campaigns", ascending=False)
    .round(2))
display(channel_summary)

,rows,average_spend_eur,average_impressions,average_website_visits,average_demo_requests,average_mqls,median_cost_per_lead_eur
campaign_flag,,,,,,,
0,98060,0.000,0.000,252.110,3.550,4.060,NaN
1,21940,"4,066.490","85,538.110","2,331.890",48.000,166.710,24.740


,campaigns,average_spend_eur,average_ctr_pct,average_mqls
campaign_channel,,,,
Email,5064,"4,050.350",2.420,166.220
Paid Search,4981,"4,068.690",2.420,167.520
Partner,3951,"4,063.980",2.420,168.420
Content,2855,"4,004.310",2.420,163.570
Webinar,2819,"4,112.000",2.430,167.370
Trade Fair,2270,"4,123.690",2.410,166.210


## 17. Data quality and join-coverage validation

The final checks cover:

- expected grain and row counts
- unique composite keys
- canonical dimensions
- allowed missing values
- numeric ranges and campaign rules
- no pre-launch activity
- and enrichment coverage for every existing sales row

In [29]:
signal_key = ["year_month", "product_group", "region_id"]
activity_key = ["year_month", "product_id", "region_id"]

expected_signal_rows = (
    len(months) * len(product_groups) * len(region_ids)
)
expected_activity_rows = len(activity_panel)

signal_checks = pd.Series({
    "expected_row_count": len(market_signals) == expected_signal_rows,
    "unique_composite_key": not market_signals.duplicated(
        signal_key
    ).any(),
    "no_missing_values": not market_signals.isna().any().any(),
    "all_months_present": set(market_signals["year_month"]) == set(months),
    "canonical_product_groups": set(
        market_signals["product_group"]
    ) == set(product_groups),
    "all_regions_present": set(
        market_signals["region_id"]
    ) == set(region_ids),
    "binary_shock_flag": market_signals[
        "demand_shock_flag"
    ].isin([0, 1]).all(),
    "demand_in_range": market_signals[
        "market_demand_index"
    ].between(
        SIGNAL_ASSUMPTIONS["market_demand_min"],
        SIGNAL_ASSUMPTIONS["market_demand_max"],
    ).all(),
    "opportunity_in_range": market_signals[
        "market_opportunity_score"
    ].between(0, 100).all(),
}, name="passed")

allowed_activity_missing = {"cost_per_lead_eur"}
unexpected_missing = (
    market_activity.drop(
        columns=list(allowed_activity_missing)
    ).isna().any().any()
)

activity_with_launch = market_activity.merge(
    product_info[[
        "product_id", "launch_month", "effective_launch_month",
    ]],
    on="product_id",
    how="left",
    validate="m:1",
)
no_prelaunch_activity = (
    activity_with_launch["effective_launch_month"].isna()
    | (
        activity_with_launch["year_month"]
        >= activity_with_launch["effective_launch_month"]
    )
).all()

activity_checks = pd.Series({
    "expected_row_count": len(market_activity) == expected_activity_rows,
    "unique_composite_key": not market_activity.duplicated(
        activity_key
    ).any(),
    "no_unexpected_missing_values": not unexpected_missing,
    "binary_campaign_flag": market_activity[
        "campaign_flag"
    ].isin([0, 1]).all(),
    "no_activity_before_effective_launch": no_prelaunch_activity,
    "zero_spend_without_campaign": market_activity.loc[
        market_activity["campaign_flag"].eq(0),
        "campaign_spend_eur",
    ].eq(0).all(),
    "zero_impressions_without_campaign": market_activity.loc[
        market_activity["campaign_flag"].eq(0),
        "campaign_impressions",
    ].eq(0).all(),
    "zero_clicks_without_campaign": market_activity.loc[
        market_activity["campaign_flag"].eq(0),
        "campaign_clicks",
    ].eq(0).all(),
    "clicks_not_above_impressions": (
        market_activity["campaign_clicks"]
        <= market_activity["campaign_impressions"]
    ).all(),
    "ctr_in_range": market_activity[
        "campaign_ctr_pct"
    ].between(0, 100).all(),
    "page_views_not_below_visits": (
        market_activity["product_page_views"]
        >= market_activity["website_visits"]
    ).all(),
    "mqls_not_below_demos": (
        market_activity["marketing_qualified_leads"]
        >= market_activity["demo_requests"]
    ).all(),
    "numeric_values_nonnegative": (
        market_activity.select_dtypes(include=np.number)
        .ge(0)
        .where(
            market_activity.select_dtypes(
                include=np.number
            ).notna(),
            True,
        )
        .all()
        .all()
    ),
}, name="passed")

display(
    pd.concat(
        {
            "market_signals": signal_checks,
            "market_activity": activity_checks,
        }
    )
    .rename("passed")
    .rename_axis(["dataset", "check"])
    .reset_index()
)

if not signal_checks.all() or not activity_checks.all():
    raise AssertionError("At least one generated-data quality check failed.")


,dataset,check,passed
0,market_signals,expected_row_count,True
1,market_signals,unique_composite_key,True
2,market_signals,no_missing_values,True
3,market_signals,all_months_present,True
4,market_signals,canonical_product_groups,True
5,market_signals,all_regions_present,True
6,market_signals,binary_shock_flag,True
7,market_signals,demand_in_range,True
8,market_signals,opportunity_in_range,True
9,market_activity,expected_row_count,True


In [30]:
# Check sales-row enrichment coverage
sales_keys = sales[[
    "year_month", "product_id", "region_id",
]].merge(
    product_info[["product_id", "product_group", "launch_month",
        "first_sales_month", "effective_launch_month",
        "launch_date_issue_flag"]],
    on="product_id",
    how="left",
    validate="m:1")

sales_before_configured_launch = (
    sales_keys["launch_month"].notna()
    & (sales_keys["year_month"] < sales_keys["launch_month"]))
sales_before_effective_launch = (
    sales_keys["effective_launch_month"].notna()
    & (
        sales_keys["year_month"]
        < sales_keys["effective_launch_month"]))
if sales_before_effective_launch.any():
    raise ValueError(
        f"{int(sales_before_effective_launch.sum()):,} sales rows occur "
        "before the effective launch month. Check product IDs and "
        "first-sales derivation.")

launch_coverage_summary = pd.Series({
    "sales_rows_before_configured_launch": int(
        sales_before_configured_launch.sum()),
    "sales_rows_before_effective_launch": int(
        sales_before_effective_launch.sum()),
    "affected_products": int(
        product_info["launch_date_issue_flag"].sum())}, name="rows")
display(launch_coverage_summary.to_frame())

signal_match = sales_keys.merge(
    market_signals[
        signal_key + ["market_demand_index"]
    ],
    on=signal_key,
    how="left",
    validate="m:1")
signal_coverage = signal_match[
    "market_demand_index"
].notna().mean()

activity_match = sales_keys.merge(
    market_activity[
        activity_key + ["website_visits"]
    ],
    on=activity_key,
    how="left",
    validate="m:1")
activity_coverage = activity_match["website_visits"].notna().mean()

unmatched_signal_keys = (
    signal_match.loc[
        signal_match["market_demand_index"].isna(),
        signal_key,
    ]
    .drop_duplicates()
    .sort_values(signal_key)
    .reset_index(drop=True))
unmatched_activity_keys = (
    activity_match.loc[
        activity_match["website_visits"].isna(),
        activity_key]
    .drop_duplicates()
    .sort_values(activity_key)
    .reset_index(drop=True))

coverage_summary = pd.DataFrame({
    "dataset": ["market_signals", "market_activity"],
    "sales_join_coverage_pct": [
        round(signal_coverage * 100, 2),
        round(activity_coverage * 100, 2)]})
display(coverage_summary)

if not unmatched_signal_keys.empty:
    print("Unmatched market-signal keys")
    display(unmatched_signal_keys.head(20))
    unmatched_signal_keys.to_csv(
        OUTPUT_DIR / "unmatched_market_signal_sales_keys.csv",
        index=False)

if not unmatched_activity_keys.empty:
    print("Unmatched market-activity keys")
    display(unmatched_activity_keys.head(20))
    unmatched_activity_keys.to_csv(
        OUTPUT_DIR / "unmatched_market_activity_sales_keys.csv",
        index=False)

if not np.isclose(signal_coverage, 1.0):
    raise ValueError(
        "Market-signal coverage is below 100%. See "
        "unmatched_market_signal_sales_keys.csv.")
if not np.isclose(activity_coverage, 1.0):
    raise ValueError(
        "Market-activity coverage is below 100%. See "
        "unmatched_market_activity_sales_keys.csv.")
print("Every sales row can receive both enrichments.")


,rows
sales_rows_before_configured_launch,19843
sales_rows_before_effective_launch,0
affected_products,101


,dataset,sales_join_coverage_pct
0,market_signals,100.000
1,market_activity,100.000


Every sales row can receive both enrichments.


## 18. Data dictionaries

In [31]:
signal_definitions = {
    "year_month": "Calendar month in YYYY-MM format.",
    "product_family": "Canonical family from the product dimension.",
    "product_group": "Canonical group from the product dimension.",
    "region_id": "Region key matching the region dimension.",
    "market_demand_index": "Synthetic demand level; 100 is the structural baseline.",
    "market_growth_pct": "Month-over-month percentage change in market demand.",
    "competitor_pressure_index": "Synthetic mean-reverting competitive pressure from 10 to 95.",
    "seasonality_index": "Seasonal demand multiplier indexed to 100.",
    "macro_business_index": "Shared monthly macroeconomic conditions indexed to 100.",
    "supply_pressure_index": "Pressure derived from stockouts and available inventory.",
    "pipeline_interest_index": "Relative opportunity-count interest within the month.",
    "demand_shock_flag": "One when a positive or negative demand shock is simulated.",
    "market_opportunity_score": "Composite opportunity score from 0 to 100.",
    "regional_market_growth_factor": "Annual structural growth factor from the region dimension."}

activity_definitions = {
    "year_month": "Calendar month in YYYY-MM format.",
    "product_id": "Product key matching the product dimension.",
    "region_id": "Region key matching the region dimension.",
    "campaign_flag": "One when a paid campaign is active.",
    "campaign_channel": "Selected campaign channel or No Active Campaign.",
    "campaign_spend_eur": "Synthetic paid campaign spend in EUR.",
    "campaign_impressions": "Synthetic paid impressions.",
    "campaign_clicks": "Synthetic paid clicks.",
    "campaign_ctr_pct": "Campaign clicks divided by impressions, expressed as percent.",
    "website_visits": "Synthetic organic and campaign-driven website visits.",
    "product_page_views": "Synthetic product-detail page views.",
    "demo_requests": "Synthetic demo requests.",
    "marketing_qualified_leads": "Demo, click, and organic qualified leads.",
    "cost_per_lead_eur": "Spend divided by MQLs; missing when spend or leads are zero.",
    "regional_crm_activity_index": "Relative regional CRM activity; 100 is monthly average.",
    "pipeline_interest_index": "Relative group-region pipeline interest."}

market_signals_dictionary = pd.DataFrame({
    "column": market_signals.columns,
    "definition": [
        signal_definitions[column]
        for column in market_signals.columns]})
market_activity_dictionary = pd.DataFrame({
    "column": market_activity.columns,
    "definition": [
        activity_definitions[column]
        for column in market_activity.columns]})

print("Market signals dictionary")
display(market_signals_dictionary)
print("Market activity dictionary")
display(market_activity_dictionary)


Market signals dictionary


,column,definition
0,year_month,Calendar month in YYYY-MM format.
1,product_family,Canonical family from the product dimension.
2,product_group,Canonical group from the product dimension.
3,region_id,Region key matching the region dimension.
4,market_demand_index,Synthetic demand level; 100 is the structural ...
5,market_growth_pct,Month-over-month percentage change in market d...
6,competitor_pressure_index,Synthetic mean-reverting competitive pressure ...
7,seasonality_index,Seasonal demand multiplier indexed to 100.
8,macro_business_index,Shared monthly macroeconomic conditions indexe...
9,supply_pressure_index,Pressure derived from stockouts and available ...


Market activity dictionary


,column,definition
0,year_month,Calendar month in YYYY-MM format.
1,product_id,Product key matching the product dimension.
2,region_id,Region key matching the region dimension.
3,campaign_flag,One when a paid campaign is active.
4,campaign_channel,Selected campaign channel or No Active Campaign.
5,campaign_spend_eur,Synthetic paid campaign spend in EUR.
6,campaign_impressions,Synthetic paid impressions.
7,campaign_clicks,Synthetic paid clicks.
8,campaign_ctr_pct,"Campaign clicks divided by impressions, expres..."
9,website_visits,Synthetic organic and campaign-driven website ...


## 19. Export

In [32]:
quality_summary = pd.DataFrame({
    "dataset": ["market_signals", "market_activity"],
    "rows": [len(market_signals), len(market_activity)],
    "columns": [
        market_signals.shape[1],
        market_activity.shape[1]],
    "duplicate_keys": [
        int(market_signals.duplicated(signal_key).sum()),
        int(market_activity.duplicated(activity_key).sum())],
    "missing_values": [
        int(market_signals.isna().sum().sum()),
        int(market_activity.isna().sum().sum())],
    "intentional_missing_cost_per_lead": [
        0,
        int(market_activity["cost_per_lead_eur"].isna().sum())],
    "sales_join_coverage_pct": [
        round(signal_coverage * 100, 2),
        round(activity_coverage * 100, 2)]})
display(quality_summary)

generation_metadata = {
    "random_seed": RANDOM_SEED,
    "market_signal_seed": MARKET_SIGNAL_SEED,
    "market_activity_seed": MARKET_ACTIVITY_SEED,
    "generation_horizon": {
        "start_month": months[0],
        "end_month": months[-1],
        "month_count": len(months),
        "source": "observed sales horizon"},
    "dimensions": {
        "products": len(product_ids),
        "product_groups": len(product_groups),
        "regions": len(region_ids)},
    "launch_date_quality": {
        "affected_products": int(
            product_info["launch_date_issue_flag"].sum()),
        "sales_rows_before_configured_launch": int(
            sales_before_configured_launch.sum()),
        "sales_rows_before_effective_launch": int(
            sales_before_effective_launch.sum()),
        "sales_keys_added_to_activity_panel": int(
            len(sales_keys_missing_from_eligibility)),
        "eligibility_rule": (
            "earlier of configured launch month and "
            "first observed sales month")},
    "outputs": {
        "market_signals": list(market_signals.shape),
        "market_activity": list(market_activity.shape)},
    "signal_assumptions": SIGNAL_ASSUMPTIONS,
    "activity_assumptions": ACTIVITY_ASSUMPTIONS,
    "quality": {
        "signal_checks": signal_checks.to_dict(),
        "activity_checks": activity_checks.to_dict(),
        "signal_coverage": float(signal_coverage),
        "activity_coverage": float(activity_coverage),
    },
}


,dataset,rows,columns,duplicate_keys,missing_values,intentional_missing_cost_per_lead,sales_join_coverage_pct
0,market_signals,6000,14,0,0,0,100.000
1,market_activity,120000,16,0,98060,98060,100.000


In [33]:
datasets_to_export = {
    "market_signals.csv": market_signals,
    "market_activity.csv": market_activity,
    "market_signals_dictionary.csv": market_signals_dictionary,
    "market_activity_dictionary.csv": market_activity_dictionary,
    "market_generation_quality_summary.csv": quality_summary,
    "product_launch_date_issues.csv": product_launch_issues,
    "activity_panel_coverage_safeguard_keys.csv": (sales_keys_missing_from_eligibility)}

for filename, dataset in datasets_to_export.items():
    dataset.to_csv(OUTPUT_DIR / filename, index=False)
with open(
    OUTPUT_DIR / "market_generation_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(generation_metadata, file, indent=2, default=str)

# samples for Github
market_signals.sample(
    min(1_000, len(market_signals)),
    random_state=RANDOM_SEED,
).sort_values(signal_key).to_csv(
    SAMPLE_OUTPUT_DIR / "market_signals_sample.csv",
    index=False)
market_activity.sample(
    min(1_000, len(market_activity)),
    random_state=RANDOM_SEED,
).sort_values(activity_key).to_csv(
    SAMPLE_OUTPUT_DIR / "market_activity_sample.csv",
    index=False)

In [34]:
# Check the output
output_manifest = pd.DataFrame([
    {
        "file": path.name,
        "folder": path.parent.name,
        "size_kb": round(path.stat().st_size / 1024, 1)}
    for folder in [OUTPUT_DIR, SAMPLE_OUTPUT_DIR]
    for path in sorted(folder.glob("*"))
    if path.is_file()])
display(output_manifest)

,file,folder,size_kb
0,activity_panel_coverage_safeguard_keys.csv,new_generated_data,0.000
1,market_activity.csv,new_generated_data,"8,996.200"
2,market_activity_dictionary.csv,new_generated_data,1.000
3,market_generation_metadata.json,new_generated_data,2.600
4,market_generation_quality_summary.csv,new_generated_data,0.200
5,market_signals.csv,new_generated_data,564.500
6,market_signals_dictionary.csv,new_generated_data,1.000
7,product_launch_date_issues.csv,new_generated_data,8.300
8,unmatched_market_activity_sales_keys.csv,new_generated_data,188.800
9,market_activity_sample.csv,02_new_generated_data,75.300


In [ ]:
# Export generated market tables to SQL Server / SSMS
import os
from decimal import Decimal
from urllib.parse import quote_plus

from sqlalchemy import create_engine, text
from sqlalchemy.types import (
    BigInteger,
    Boolean,
    Integer,
    Numeric,
    String,
)

# Connection settings
SQL_SERVER = os.getenv("SQL_SERVER", r"localhost")
SQL_DATABASE = "RevenueAnalytics"
SQL_SCHEMA = "dbo"
SQL_DRIVER = "ODBC Driver 18 for SQL Server"
SQL_IF_EXISTS = "replace"

odbc_connection_string = (
    f"DRIVER={{{SQL_DRIVER}}};"
    f"SERVER={SQL_SERVER};"
    f"DATABASE={SQL_DATABASE};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect="
    + quote_plus(odbc_connection_string),
    # More reliable when a numeric column contains both values and NULLs.
    fast_executemany=False,
)

# Explicit SQL Server datatypes prevent decimal values from being
# interpreted as text and preserve the intended table schemas.
sql_dtypes = {
    "factMarketSignals": {
        "year_month": String(7),
        "product_family": String(100),
        "product_group": String(100),
        "region_id": Integer(),
        "market_demand_index": Numeric(9, 3),
        "market_growth_pct": Numeric(9, 3),
        "competitor_pressure_index": Numeric(9, 3),
        "seasonality_index": Numeric(9, 3),
        "macro_business_index": Numeric(9, 3),
        "supply_pressure_index": Numeric(9, 3),
        "pipeline_interest_index": Numeric(9, 3),
        "demand_shock_flag": Boolean(),
        "market_opportunity_score": Numeric(9, 3),
        "regional_market_growth_factor": Numeric(9, 6),
    },
    "factMarketActivities": {
        "year_month": String(7),
        "product_id": Integer(),
        "region_id": Integer(),
        "campaign_flag": Boolean(),
        "campaign_channel": String(50),
        "campaign_spend_eur": Numeric(18, 2),
        "campaign_impressions": BigInteger(),
        "campaign_clicks": BigInteger(),
        "campaign_ctr_pct": Numeric(9, 3),
        "website_visits": BigInteger(),
        "product_page_views": BigInteger(),
        "demo_requests": Integer(),
        "marketing_qualified_leads": Integer(),
        "cost_per_lead_eur": Numeric(18, 2),
        "regional_crm_activity_index": Numeric(9, 3),
        "pipeline_interest_index": Numeric(9, 3),
    },
}

sql_tables = {
    "factMarketSignals": market_signals,
    "factMarketActivities": market_activity,
}

sql_key_columns = {
    "factMarketSignals": [
        "year_month", "product_group", "region_id"
    ],
    "factMarketActivities": [
        "year_month", "product_id", "region_id"
    ],
}

sql_integer_columns = {
    "factMarketSignals": ["region_id"],
    "factMarketActivities": [
        "product_id",
        "region_id",
        "campaign_impressions",
        "campaign_clicks",
        "website_visits",
        "product_page_views",
        "demo_requests",
        "marketing_qualified_leads",
    ],
}

sql_boolean_columns = {
    "factMarketSignals": ["demand_shock_flag"],
    "factMarketActivities": ["campaign_flag"],
}

sql_decimal_columns = {
    table_name: [
        column_name
        for column_name, sql_type in dtype_map.items()
        if isinstance(sql_type, Numeric)
    ]
    for table_name, dtype_map in sql_dtypes.items()
}


def _python_int_or_none(value):
    if pd.isna(value):
        return None
    return int(value)


def _python_decimal_or_none(value):
    if pd.isna(value):
        return None

    decimal_value = Decimal(str(value))
    if not decimal_value.is_finite():
        raise ValueError(f"Non-finite numeric value encountered: {value}")
    return decimal_value


def _python_bool_or_none(value):
    if pd.isna(value):
        return None
    return bool(value)


def prepare_market_table_for_sql(table_name, source_df):
    expected_columns = list(sql_dtypes[table_name])
    missing_columns = sorted(set(expected_columns) - set(source_df.columns))
    unexpected_columns = sorted(set(source_df.columns) - set(expected_columns))

    if missing_columns or unexpected_columns:
        raise ValueError(
            f"Schema mismatch for {table_name}. "
            f"Missing={missing_columns}; unexpected={unexpected_columns}"
        )

    df = source_df.loc[:, expected_columns].copy()

    duplicate_count = int(df.duplicated(sql_key_columns[table_name]).sum())
    if duplicate_count:
        raise ValueError(
            f"{table_name} contains {duplicate_count:,} duplicate key rows."
        )

    for column in sql_integer_columns[table_name]:
        numeric = pd.to_numeric(df[column], errors="raise")
        non_null = numeric.dropna()
        if not ((non_null % 1) == 0).all():
            raise ValueError(
                f"{table_name}.{column} contains non-integer values."
            )
        df[column] = pd.Series(
            [_python_int_or_none(value) for value in numeric],
            index=df.index,
            dtype=object,
        )

    for column in sql_boolean_columns[table_name]:
        values = df[column]
        if pd.api.types.is_bool_dtype(values):
            converted = values
        else:
            normalized = values.astype(str).str.strip().str.lower()
            converted = normalized.map({
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "nan": None,
                "none": None,
                "": None,
            })
            invalid_mask = converted.isna() & values.notna()
            if invalid_mask.any():
                bad_values = values.loc[invalid_mask].unique()[:5]
                raise ValueError(
                    f"{table_name}.{column} contains invalid BIT values: "
                    f"{bad_values}"
                )
        df[column] = pd.Series(
            [_python_bool_or_none(value) for value in converted],
            index=df.index,
            dtype=object,
        )

    for column in sql_decimal_columns[table_name]:
        values = df[column]
        if pd.api.types.is_object_dtype(values):
            values = (
                values.astype(str)
                .str.strip()
                .str.replace(",", ".", regex=False)
                .replace({"nan": None, "None": None, "": None})
            )

        numeric = pd.to_numeric(values, errors="coerce")
        original_values = pd.Series(values, index=df.index)
        invalid_mask = numeric.isna() & original_values.notna()
        if invalid_mask.any():
            bad_values = original_values.loc[invalid_mask].unique()[:5]
            raise ValueError(
                f"{table_name}.{column} contains invalid numeric values: "
                f"{bad_values}"
            )

        df[column] = pd.Series(
            [_python_decimal_or_none(value) for value in numeric],
            index=df.index,
            dtype=object,
        )

    return df


# Confirm the target database before writing any table.
with engine.connect() as connection:
    connected_database = connection.execute(
        text("SELECT DB_NAME()")
    ).scalar_one()

if connected_database != SQL_DATABASE:
    raise RuntimeError(
        f"Connected to unexpected database '{connected_database}'; "
        f"expected '{SQL_DATABASE}'."
    )

print(
    f"Connected to SQL Server '{SQL_SERVER}', "
    f"database '{connected_database}'."
)

sql_import_rows = []

for table_name, source_df in sql_tables.items():
    sql_df = prepare_market_table_for_sql(table_name, source_df)
    null_cells = int(sql_df.isna().sum().sum())

    print(
        f"Uploading {table_name}: {len(sql_df):,} rows -> "
        f"{SQL_DATABASE}.{SQL_SCHEMA}.{table_name} "
        f"(NULL cells: {null_cells:,})"
    )

    sql_df.to_sql(
        name=table_name,
        con=engine,
        schema=SQL_SCHEMA,
        if_exists=SQL_IF_EXISTS,
        index=False,
        dtype=sql_dtypes[table_name],
        chunksize=1000,
    )

    with engine.connect() as connection:
        sql_row_count = connection.execute(
            text(
                f"SELECT COUNT_BIG(*) "
                f"FROM [{SQL_SCHEMA}].[{table_name}]"
            )
        ).scalar_one()

    if int(sql_row_count) != len(sql_df):
        raise AssertionError(
            f"Row-count mismatch for {table_name}: "
            f"DataFrame={len(sql_df):,}, SQL={int(sql_row_count):,}"
        )

    sql_import_rows.append({
        "table": table_name,
        "dataframe_rows": len(sql_df),
        "sql_rows": int(sql_row_count),
        "database": SQL_DATABASE,
        "schema": SQL_SCHEMA,
        "status": "OK",
    })

sql_import_summary = pd.DataFrame(sql_import_rows)
display(sql_import_summary)

print(
    f"SQL import completed successfully: "
    f"{len(sql_import_summary)} tables loaded into "
    f"{SQL_DATABASE}.{SQL_SCHEMA}."
)

engine.dispose()

C:\Users\Anast\AppData\Local\Temp\ipykernel_38100\2652520710.py:235: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as connection:


Connected to SQL Server 'localhost', database 'RevenueAnalytics'.
Uploading factMarketSignals: 6,000 rows -> RevenueAnalytics.dbo.factMarketSignals (NULL cells: 0)
Uploading factMarketActivities: 120,000 rows -> RevenueAnalytics.dbo.factMarketActivities (NULL cells: 98,060)


,table,dataframe_rows,sql_rows,database,schema,status
0,factMarketSignals,6000,6000,RevenueAnalytics,dbo,OK
1,factMarketActivities,120000,120000,RevenueAnalytics,dbo,OK


SQL import completed successfully: 2 tables loaded into RevenueAnalytics.dbo.
